# ViC-MAE: ViT-B/16 Finetuning on Tiny-ImageNet-200

**Paper:** ViC-MAE (ECCV 2024) — Hernandez, Villegas, Ordonez  
**Repo:** https://github.com/jeffhernandez1995/ViC-MAE  

This notebook finetunes a ViC-MAE pretrained ViT-B/16 on Tiny-ImageNet-200 (200 classes).  

**Before running:**
1. Settings → Accelerator → **GPU T4 x2** (or P100)
2. Settings → Internet → **On**
3. Run all cells top to bottom (Runtime → Run All)

## 1. Environment Setup

In [ ]:
import subprocess, sys

# Install correct timm version (0.4.12 required for MAE-style ViT)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "timm==0.4.12", "wandb", "matplotlib"])

import torch
print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM: {vram_gb:.1f} GB")

## 2. Clone Repo & Download Assets

In [ ]:
import os, urllib.request, zipfile, time

WORK_DIR = "/kaggle/working"
REPO_DIR = os.path.join(WORK_DIR, "ViC-MAE")
CKPT_PATH = os.path.join(WORK_DIR, "vicmae_pretrain_vit_base_in1k_k400.pth")
DATA_DIR = os.path.join(WORK_DIR, "tiny-imagenet-200")
ZIP_PATH = os.path.join(WORK_DIR, "tiny-imagenet-200.zip")

# Clone repo
if not os.path.exists(REPO_DIR):
    os.system(f"git clone https://github.com/jeffhernandez1995/ViC-MAE {REPO_DIR}")
    print("\u2705 Repo cloned")
else:
    print("\u2705 Repo already exists")

# Download checkpoint (~327 MB)
CKPT_URL = "https://huggingface.co/jeffhernandez1995/vicmae/resolve/main/vicmae_pretrain_vit_base_in1k_k400.pth"
if not os.path.exists(CKPT_PATH):
    print("Downloading ViT-B/16 checkpoint...")
    urllib.request.urlretrieve(CKPT_URL, CKPT_PATH)
    size_mb = os.path.getsize(CKPT_PATH) / 1e6
    print(f"\u2705 Checkpoint downloaded ({size_mb:.1f} MB)")
else:
    print("\u2705 Checkpoint already exists")

# Download Tiny-ImageNet (~248 MB)
TINY_URL = "http://cs231n.stanford.edu/tiny-imagenet-200.zip"
if not os.path.exists(DATA_DIR):
    if not os.path.exists(ZIP_PATH):
        print("Downloading Tiny-ImageNet-200...")
        urllib.request.urlretrieve(TINY_URL, ZIP_PATH)
    print("Extracting...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(WORK_DIR)
    os.remove(ZIP_PATH)
    print("\u2705 Tiny-ImageNet extracted")
else:
    print("\u2705 Tiny-ImageNet already exists")

## 3. Organize Dataset into ImageFolder Structure

In [ ]:
import shutil
from pathlib import Path

DATA_DIR = "/kaggle/working/tiny-imagenet-200"

# Fix train: move images out of 'images/' subdirectory
train_dir = Path(DATA_DIR) / "train"
for cls_dir in train_dir.iterdir():
    if not cls_dir.is_dir():
        continue
    images_subdir = cls_dir / "images"
    if images_subdir.exists():
        for img in images_subdir.iterdir():
            img.rename(cls_dir / img.name)
        images_subdir.rmdir()
    for txt in cls_dir.glob("*.txt"):
        txt.unlink()

# Fix val: reorganize from flat to per-class subdirectories
val_dir = Path(DATA_DIR) / "val"
annotations_file = val_dir / "val_annotations.txt"

if annotations_file.exists():
    img_to_class = {}
    with open(annotations_file) as f:
        for line in f:
            parts = line.strip().split('\t')
            img_to_class[parts[0]] = parts[1]

    images_dir = val_dir / "images"
    for img_name, cls_id in img_to_class.items():
        cls_dir = val_dir / cls_id
        cls_dir.mkdir(exist_ok=True)
        src = images_dir / img_name
        if src.exists():
            shutil.move(str(src), str(cls_dir / img_name))

    if images_dir.exists():
        shutil.rmtree(images_dir)
    annotations_file.unlink()

# Verify
train_classes = len(list(train_dir.iterdir()))
val_classes = len([d for d in val_dir.iterdir() if d.is_dir()])
train_images = sum(1 for _ in train_dir.rglob("*.JPEG"))
val_images = sum(1 for _ in val_dir.rglob("*.JPEG"))
print(f"\u2705 Train: {train_classes} classes, {train_images} images")
print(f"\u2705 Val:   {val_classes} classes, {val_images} images")

## 4. Apply Bug Fix & Run 50-Epoch Finetuning

In [ ]:
import os, torch

REPO_DIR = "/kaggle/working/ViC-MAE"
CKPT_PATH = "/kaggle/working/vicmae_pretrain_vit_base_in1k_k400.pth"
DATA_DIR = "/kaggle/working/tiny-imagenet-200"
OUTPUT_DIR = "/kaggle/working/output"

# Apply the eval bug fix (epoch variable unbound in --eval mode)
finetune_path = os.path.join(REPO_DIR, "image_finetune.py")
with open(finetune_path, 'r') as f:
    content = f.read()

if 'epoch=epoch)' in content.split('if args.eval:')[1].split('exit(0)')[0]:
    content = content.replace(
        'evaluate(data_loader_val, model, args.device, epoch=epoch)\n        print(f"Accuracy',
        'evaluate(data_loader_val, model, args.device, epoch=0)\n        print(f"Accuracy',
        1  # replace only the first occurrence (the eval block)
    )
    with open(finetune_path, 'w') as f:
        f.write(content)
    print("\u2705 Bug fix applied to image_finetune.py")
else:
    print("\u2705 Bug fix already applied")

# Determine optimal batch size based on GPU VRAM
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9

if vram_gb >= 14:       # T4 (16 GB) or P100 (16 GB)
    BATCH_SIZE = 128
    ACCUM_ITER = 1
elif vram_gb >= 10:     # 12 GB GPUs
    BATCH_SIZE = 64
    ACCUM_ITER = 2
else:                   # smaller GPUs
    BATCH_SIZE = 32
    ACCUM_ITER = 4

print(f"GPU VRAM: {vram_gb:.1f} GB -> batch_size={BATCH_SIZE}, accum_iter={ACCUM_ITER}")
print(f"Effective batch size: {BATCH_SIZE * ACCUM_ITER}")

# Launch finetuning
os.environ["WANDB_MODE"] = "disabled"

cmd = f"""cd {REPO_DIR} && python image_finetune.py \\
    --batch_size {BATCH_SIZE} \\
    --accum_iter {ACCUM_ITER} \\
    --epochs 50 \\
    --model vit_base_patch16 \\
    --finetune {CKPT_PATH} \\
    --data_path {DATA_DIR} \\
    --nb_classes 200 \\
    --output_dir {OUTPUT_DIR} \\
    --log_dir {OUTPUT_DIR} \\
    --blr 1e-3 \\
    --layer_decay 0.75 \\
    --weight_decay 0.05 \\
    --drop_path 0.1 \\
    --mixup 0.8 \\
    --cutmix 1.0 \\
    --reprob 0.25 \\
    --warmup_epochs 5 \\
    --smoothing 0.1 \\
    --input_size 224 \\
    --num_workers 4 \\
    --pin_mem"""

print(f"\n\U0001f680 Starting 50-epoch finetuning...")
print(f"   Output: {OUTPUT_DIR}")
os.system(cmd)
print("\n\u2705 Finetuning complete!")

## 5. Evaluate & Plot Results

In [ ]:
import json, os, glob, csv
import matplotlib.pyplot as plt

OUTPUT_DIR = "/kaggle/working/output"

# Find the timestamped output directory
subdirs = sorted(glob.glob(os.path.join(OUTPUT_DIR, "*vit_base*")))
if not subdirs:
    log_file = os.path.join(OUTPUT_DIR, "log.txt")
    run_dir = OUTPUT_DIR
else:
    run_dir = subdirs[-1]
    log_file = os.path.join(run_dir, "log.txt")

print(f"Run directory: {run_dir}")
print(f"Log file: {log_file}")

# Parse log.txt
epochs, train_loss, val_top1, val_top5 = [], [], [], []
with open(log_file) as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        try:
            d = json.loads(line)
            if "epoch" in d and "train_loss" in d:
                epochs.append(d["epoch"])
                train_loss.append(d["train_loss"])
                val_top1.append(d.get("test_acc1"))
                val_top5.append(d.get("test_acc5"))
        except json.JSONDecodeError:
            pass

print(f"\nParsed {len(epochs)} epochs")
print(f"Final train loss: {train_loss[-1]:.4f}")
print(f"Final val top-1:  {val_top1[-1]:.2f}%")
print(f"Final val top-5:  {val_top5[-1]:.2f}%")
print(f"Best val top-1:   {max(val_top1):.2f}%")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs, train_loss, 'b-o', markersize=3, label='Train Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, val_top1, 'g-o', markersize=3, label='Top-1 Accuracy')
axes[1].plot(epochs, val_top5, 'r-s', markersize=3, label='Top-5 Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Validation Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('ViC-MAE ViT-B/16 Finetuning on Tiny-ImageNet-200', fontsize=14, fontweight='bold')
plt.tight_layout()

plot_path = os.path.join(run_dir, "training_curves.png")
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"\nPlot saved: {plot_path}")

# Save CSV
csv_path = os.path.join(run_dir, "training_log.csv")
with open(csv_path, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['epoch', 'train_loss', 'val_top1', 'val_top5'])
    for e, tl, v1, v5 in zip(epochs, train_loss, val_top1, val_top5):
        writer.writerow([e, f"{tl:.4f}", f"{v1:.2f}", f"{v5:.2f}"])
print(f"CSV saved: {csv_path}")

## 6. Package Results for Download

In [ ]:
import shutil, glob, os

OUTPUT_DIR = "/kaggle/working/output"
subdirs = sorted(glob.glob(os.path.join(OUTPUT_DIR, "*vit_base*")))
run_dir = subdirs[-1] if subdirs else OUTPUT_DIR

zip_name = "/kaggle/working/vicmae_results"
shutil.make_archive(zip_name, 'zip', run_dir)
print(f"Results packaged: {zip_name}.zip")
print(f"Download from the Output tab on the right sidebar.")

print(f"\nFiles in {run_dir}:")
for f in sorted(os.listdir(run_dir)):
    size = os.path.getsize(os.path.join(run_dir, f)) / 1e6
    print(f"   {f:40s} {size:8.1f} MB")